In [1]:
import os
import pandas as pd
from openai import OpenAI

import numpy as np
import json
from dotenv import load_dotenv

In [2]:
repo_dir = "/Users/haya1/Documents/LanguageModel_Labels/congressional_bills"
os.chdir(repo_dir)

# Place API_KEY in the .env file
load_dotenv()
api_key = os.environ.get('API_KEY')
client = OpenAI(api_key=api_key)

## Generate prompts

In [3]:
data_dir = os.path.join(repo_dir, "01_bills")
bills = pd.read_csv(os.path.join(data_dir, "bills.csv"))[['BillID', 'Major', 'MajorText', 'Description']]

llm_dir = os.path.join(repo_dir, "02_llm")
# base_prompt = open(os.path.join(llm_dir, "base_prompt_json.txt"), 'r').read()

In [4]:
prompting_strategies = pd.DataFrame(data={
    "Name": [
        "No Modification", 
        "Persona Modification", "Persona Modification", "Persona Modification", "Persona Modification", 
        "Chain-of-Thoughts Prompting", "Chain-of-Thoughts Prompting", "Chain-of-Thoughts Prompting", 
        "Few-Shot Prompting", "Few-Shot Prompting", "Few-Shot Prompting"
        ],
    "BeforeQuestion": [
        "",
        "You are a knowledgeable political analyst. ",
        "Answer this question as if you are a political scientist that studies legislation in the United States Congress. ",
        "Answer this question as if you are an expert in United States politics. ",
        "Answer this question as if you were a helpful research assistant for a political scientist. ",
        "", "", "",
        "", "", "",
        ],
    "BeforeAnswer": [
        "", 
        "", "", "", "", 
        "Think carefully. ", 
        "Let's think step by step. Lay out each step. ", 
        "Please provide an explanation for your answer. ",
        "", "", "",
    ],
    "Explanation":[
        "", 
        "", "", "", "",
        ',\n    "Explanation": a one-sentence explanation of your bill category answer',
        ',\n    "Explanation": a one-sentence explanation of your bill category answer',
        ',\n    "Explanation": a one-sentence explanation of your bill category answer',
        "", "", "",
    ],
    "Examples":[
        np.nan, 
        np.nan, np.nan, np.nan,  np.nan,
        np.nan, np.nan, np.nan, 
        1, 2, 3,
    ],
    "Model":[
        "gpt-3.5-turbo",
        "gpt-3.5-turbo", "gpt-3.5-turbo", "gpt-3.5-turbo", "gpt-3.5-turbo", 
        "gpt-3.5-turbo", "gpt-3.5-turbo", "gpt-3.5-turbo", 
        "gpt-3.5-turbo", "gpt-3.5-turbo", "gpt-3.5-turbo", 
    ]
})

In [5]:
def create_message_user(description, before_question=None, before_answer=None, answer_format_explanation=None):
    base_prompt = open(os.path.join(repo_dir, "02_llm/base_prompt_json.txt"), 'r').read()
    before_question = "" if before_question is None else before_question
    before_answer = "" if before_answer is None else before_answer
    answer_format_explanation = "" if answer_format_explanation is None else answer_format_explanation

    content =  base_prompt % (before_question, description, before_answer, answer_format_explanation)
    return {"role": "user", "content": content}
# print(create_message_user(description="A bill description.")["content"])

def create_message_assistant(category, confidence, explanation=None):
    base_answer = {
        "Category": category,
        "Confidence": np.round(confidence, decimals=2)
    }

    if explanation is not None:
        base_answer["explanation"] = explanation

    prompt =  json.dumps(base_answer)
    return {"role": "assistant", "content": prompt}

def create_fewshot_examples(bills_examples_set, min_confidence=0.8, max_confidence=1):
    messages = []
    for _, bill in bills_examples_set.iterrows():
        messages.append(create_message_user(description=bill["Description"]))
        messages.append(create_message_assistant(category=bill["Major"], confidence=np.random.uniform(min_confidence, max_confidence)))
    return messages

In [6]:
# Bills used for few-shot prompting
bills_examples_path = os.path.join(llm_dir, "bills_examples.csv")
if os.path.exists(bills_examples_path):
    bills_examples = pd.read_csv(bills_examples_path)
else:
    # TODO: add seed for replication
    bills_examples = bills.groupby("Major").sample(n=1, replace=False).sample(n=15, replace=False).reset_index(drop=True)
    bills_examples["ExampleSet"] = [1,2,3] * 5
    bills_examples.to_csv(os.path.join(llm_dir, "bills_examples.csv"), index=False)

condition = bills["BillID"].apply(lambda x: x not in bills_examples["BillID"].to_list())
bills_without_examples = bills[condition]
print(f"Excluded few-shot examples, n = {len(bills_examples)}, from the data. Number of remaining bills = {len(bills_without_examples)}")
bills_without_examples = bills_without_examples.groupby('Major').sample(n=2).sample(n=5) # TODO: remove .groupby('Major').sample(n=2).sample(n=5)
print(f"Currently we test the code on only {len(bills_without_examples)} bills")
bills_without_examples.to_csv(os.path.join(llm_dir, "bills_without_examples.csv"), index=False)

Excluded few-shot examples, n = 15, from the data. Number of remaining bills = 255365
Currently we test the code on only 5 bills


In [7]:
# TODO: improve prompt creation

# # def save_examples(josnl_path, examples):
# bills_examples = pd.read_csv(os.path.join(llm_dir, "bills_examples.csv"))

# prompts = []
# for _, strategy in prompting_strategies.iterrows():
#     example_set = bills_examples[bills_examples["ExampleSet"] == strategy["ExampleSet"]]
#     message = [] if (strategy["Name"]!="Few-Shot Prompting") else create_fewshot_examples(example_set)
#     prompts.append({"Example": message})

# prompts_path = os.path.join(llm_dir, "examples01.jsonl")
# with open(prompts_path, "w") as f:
#     for prompt in prompts:
#         f.write(json.dumps(prompt) + "\n")

In [8]:
examples = pd.read_csv(os.path.join(llm_dir, "bills_examples.csv"))
bills = pd.read_csv(os.path.join(llm_dir, "bills_without_examples.csv"))

# n_prompts = len(bills) * len(prompting_strategies)
# prompt_id = list(range(1, n_prompts+1))
prompt_id = 0
prompts = []

for _, bill in bills.iterrows():
    for _, strategy in prompting_strategies.iterrows():
        
        example_set = examples[examples["ExampleSet"]== strategy["Examples"]]
        messages = [] if (strategy["Name"]!="Few-Shot Prompting") else create_fewshot_examples(example_set)
        messages.append(create_message_user(description=bill["Description"], before_question=strategy["BeforeQuestion"], before_answer=strategy["BeforeAnswer"], answer_format_explanation=strategy["Explanation"]))

        prompt_id += 1
        prompt = {
            "PromptID": prompt_id,
            "BillID": bill["BillID"],
            "Description": bill["Description"],
            "PromptingStrategy": strategy["Name"],
            "Model": strategy["Model"],
            "Major": bill["Major"],
            "MajorText": bill["MajorText"],
            "Messages": messages
        }
        prompts.append(prompt)

prompts_path = os.path.join(llm_dir, "prompts.jsonl")
with open(prompts_path, "w") as f:
    for prompt in prompts:
        f.write(json.dumps(prompt) + "\n")

In [9]:
!head -5 $prompts_path

{"PromptID": 1, "BillID": "112-HR-3970", "Description": "To suspend temporarily the duty on mixtures of Reactive Red 198 and Reactive Red 239.", "PromptingStrategy": "No Modification", "Model": "gpt-3.5-turbo", "Major": 17, "MajorText": "Foreign Trade", "Messages": [{"role": "user", "content": "Here is a description of a bill introduced in the U.S. Congress:\n\"To suspend temporarily the duty on mixtures of Reactive Red 198 and Reactive Red 239.\"\n\nPlease classify this description into one of the following categories:\n1. Macroeconomics\n2. Civil Rights, Minority Issues, and Civil Liberties\n3. Health\n4. Agriculture\n5. Labor and Employment\n6. Education\n7. Environment\n8. Energy\n9. Immigration\n10. Transportation\n11. Law, Crime, and Family Issues\n12. Social Welfare\n13. Community Development and Housing Issues\n14. Banking, Finance, and Domestic Commerce\n15. Defense\n16. Space, Science, Technology, and Communications\n17. Foreign Trade\n18. International Affairs and Foreign Ai

## Generate responses

In [10]:
# define a response function that gives us the LLM's response to a user prompt
def query_llm(messages, model, temperature=0, num_responses=1):
    response_text = client.chat.completions.create(
        model = model,
        logprobs = True,
        n = num_responses,
        temperature = temperature,
        response_format = {"type": "json_object"},
        messages = messages
    ).choices[0].message.content
    
    response_json = json.loads(response_text)
    
    if not ("Explanation" in response_json.keys()):
        response_json["Explanation"] = np.nan
    
    return response_json

In [11]:
major_text_ours = {
    1 : "Macroeconomics",
    2 : "Civil Rights, Minority Issues, and Civil Liberties",
    3 : "Health",
    4 : "Agriculture",
    5 : "Labor and Employment",
    6 : "Education",
    7 : "Environment",
    8 : "Energy",
    9 : "Immigration",
    10: "Transportation",
    11: "Law, Crime, and Family Issues",
    12: "Social Welfare",
    13: "Community Development and Housing Issues",
    14: "Banking, Finance, and Domestic Commerce",
    15: "Defense",
    16: "Space, Science, Technology, and Communications",
    17: "Foreign Trade",
    18: "International Affairs and Foreign Aid",
    19: "Government Operations",
    20: "Public Lands and Water Management",
    21: "Arts and Entertainment"
}

In [12]:
prompts = pd.read_json(path_or_buf=prompts_path, lines=True)
print(f"Loaded prompts data, n = {len(prompts)}")

major_llm = []
confidence_llm = []
explanation_llm = []
for _, prompt in prompts.iterrows():
    response = query_llm(messages=prompt["Messages"], 
                         model=prompt["Model"], 
                         temperature=0)
    major_llm.append(response["Category"])
    confidence_llm.append(response["Confidence"])
    explanation_llm.append(response["Explanation"])

responses = prompts.loc[:, prompts.columns != "Messages"]
responses["MajorLLM"] = major_llm
responses["MajorTextLLM"] = responses["MajorLLM"].apply(lambda x: major_text_ours[x] if pd.notnull(x) else x)
responses["ConfidenceLLM"] = confidence_llm
responses["ExplanationLLM"] = explanation_llm
responses.head()

Loaded prompts data, n = 55


,PromptID,BillID,Description,PromptingStrategy,Model,Major,MajorText,MajorLLM,MajorTextLLM,ConfidenceLLM,ExplanationLLM
0,1,112-HR-3970,To suspend temporarily the duty on mixtures of...,No Modification,gpt-3.5-turbo,17,Foreign Trade,14,"Banking, Finance, and Domestic Commerce",0.75,NaN
1,2,112-HR-3970,To suspend temporarily the duty on mixtures of...,Persona Modification,gpt-3.5-turbo,17,Foreign Trade,14,"Banking, Finance, and Domestic Commerce",0.70,NaN
2,3,112-HR-3970,To suspend temporarily the duty on mixtures of...,Persona Modification,gpt-3.5-turbo,17,Foreign Trade,14,"Banking, Finance, and Domestic Commerce",0.75,NaN
3,4,112-HR-3970,To suspend temporarily the duty on mixtures of...,Persona Modification,gpt-3.5-turbo,17,Foreign Trade,14,"Banking, Finance, and Domestic Commerce",0.75,NaN
4,5,112-HR-3970,To suspend temporarily the duty on mixtures of...,Persona Modification,gpt-3.5-turbo,17,Foreign Trade,14,"Banking, Finance, and Domestic Commerce",0.75,NaN


In [13]:
responses.to_csv(os.path.join(llm_dir, "responses.csv"), index=False)

In [14]:
responses = pd.read_csv(os.path.join(llm_dir, "responses.csv"))